In [24]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import pickle

In [25]:
model = tf.keras.models.load_model('model.h5')

# load encoder and scaler
with open('onehot_encoder_geo.pkl', 'rb') as f:
    onehot_encoder_geo = pickle.load(f) 

with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)


In [26]:
input_data = { 
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}
input_df = pd.DataFrame([input_data])

In [27]:
# Encode Gender
input_df['Gender'] = label_encoder.transform(input_df['Gender'])

# One-hot encode Geography
geo_encoded = onehot_encoder_geo.transform(
    input_df[['Geography']]
)

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

# Remove original Geography column
input_df.drop(columns=['Geography'], inplace=True)

# Combine the encoded columns
input_df = pd.concat(
    [input_df.reset_index(drop=True),
     geo_encoded_df.reset_index(drop=True)],
    axis=1
)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [28]:
input_scaled = scaler.transform(input_df)

In [38]:
prediction = model.predict(input_scaled)

probability = prediction[0][0]

print("Churn probability:", probability)
print("Prediction:", "Churn" if probability >= 0.5 else "No Churn")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
Churn probability: 0.039852094
Prediction: No Churn
